# 12 - Webhooks and Real-Time Communication

## 1. Webhooks

**Definition:** Usually your backend calls other services. With a webhook it's the other way round: **another service calls your backend** when something happens. You give it a URL in advance, and it sends a POST the moment the event occurs.

**Analogy:** Polling = calling the restaurant every 5 minutes asking "is my food ready?" Webhook = the restaurant calls you when it's ready.

**Polling vs Webhook:**
- **Polling:** you keep asking; most calls are wasted, updates arrive late
- **Webhook:** they tell you instantly; fewer calls

**Key points:**
- **Verify the signature:** sender signs the body with a shared secret (HMAC); reject with 401 if it doesn't match
- **Handle duplicates:** sender retries on timeout, so store the event id and skip repeats (idempotency)
- **Respond fast:** return 200 immediately, do heavy work in the background
- **Limitation:** endpoint must be public; if your server is down you can miss events, so keep a fallback check

**Real-world example:** Customer pays on Razorpay → Razorpay sends POST /webhooks/razorpay → backend verifies signature, checks event id, marks order paid, returns 200.

**Interview line:** "Webhooks let an external service push events to us instead of us polling. I verify the HMAC signature, handle retries using the event id, and return 200 quickly while processing in the background."

In [ ]:
# ==================================================
# 1. WEBHOOK RECEIVER - signature check + duplicate check
# ==================================================
import hmac, hashlib, json

WEBHOOK_SECRET = "shared-secret-with-razorpay"
processed_events = set()
orders = {"order_77": "pending"}

def razorpay_sends(event):
    body = json.dumps(event)
    signature = hmac.new(WEBHOOK_SECRET.encode(), body.encode(), hashlib.sha256).hexdigest()
    return body, signature

def receive_webhook(body, signature):
    expected = hmac.new(WEBHOOK_SECRET.encode(), body.encode(), hashlib.sha256).hexdigest()
    if not hmac.compare_digest(expected, signature):
        return 401, "Invalid signature - rejected"
    event = json.loads(body)
    if event["event_id"] in processed_events:
        return 200, "Duplicate - already processed, skipped"
    processed_events.add(event["event_id"])
    orders[event["order_id"]] = "paid"
    return 200, f"Order {event['order_id']} marked paid"

event = {"event_id": "evt_1", "type": "payment.captured", "order_id": "order_77"}
body, sig = razorpay_sends(event)

print("=" * 60)
print("[1. REAL]     ", receive_webhook(body, sig))
print("[2. RETRY]    ", receive_webhook(body, sig))
fake_body = json.dumps({"event_id": "evt_9", "type": "payment.captured", "order_id": "order_77"})
print("[3. FAKE]     ", receive_webhook(fake_body, "made-up-signature"))
print("[ORDERS]      ", orders)
print("=" * 60)

## 2. Real-Time Options

| Method | Direction | How | Example |
|---|---|---|---|
| Polling | Client asks repeatedly | New request every few seconds | Checking order status |
| Webhook | Server → server | Other service POSTs to your URL | Razorpay payment success |
| SSE | Server → client (one way) | One open HTTP connection, server streams events | Live scores, LLM streaming, LangGraph node progress |
| WebSocket | Both ways | Open two-way connection | Chat apps, multiplayer games |

**My project (AIR 2.0):** LangGraph workflow, with SSE streaming each node's progress to the UI, so users saw which step finished and which was running next.

**HATEOAS:** Responses include links to next actions (e.g. an order response with "cancel" and "pay" links). Rarely used; knowing the definition is enough.

**Interview line:** "Webhooks are server-to-server notifications, SSE is a one-way server-to-client stream, and WebSockets are a two-way live connection. I used SSE to stream LangGraph node progress in my project."

## Key Takeaways

- Webhook = another service calls you when an event happens
- Verify HMAC signature, skip duplicate event ids, return 200 fast
- Polling wastes calls; webhooks are instant
- SSE = one-way server stream; WebSocket = two-way live connection
- HATEOAS = links to next actions in responses (rarely used)